In [ ]:
#드라이브 마운트

from google.colab import drive
drive.mount('/content/drive')

In [3]:
#이건 시작할때마다 꼭 돌려야하는 임포트 문들
import zipfile
import os, json, numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, accuracy_score
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torchvision.models as models

In [ ]:
#zip 파일 압축 해제
zip_path = '/content/drive/MyDrive/processed_tensors_Hundred.zip'
extract_path = '/content'

'''
#학습시키고 싶은 동작에 따라 아래에서 골라서 다른 파일 로드
zip_path = '/content/drive/MyDrive/processed_tensors_HipCircle.zip'
extract_path = '/content'

zip_path = '/content/drive/MyDrive/processed_tensors_Swimming.zip'
extract_path = '/content'

zip_path = '/content/drive/MyDrive/processed_tensors_Teaser.zip'
extract_path = '/content'
'''

# 압축 해제
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# 압축 해제된 경로 확인
os.listdir(extract_path)


In [5]:
#이건 데이터 처리하는거
# 예측할 관절 리스트 (12개)
ALL_JOINTS = [
    "RShoulder", "RElbow", "RWrist",
    "LShoulder", "LElbow", "LWrist",
    "RHip", "RKnee", "RAnkle",
    "LHip", "LKnee", "LAnkle"
]

#관절 이름 → 다중 레이블 벡터
def joints_to_vector(wrong_joints):
    return [1 if joint in wrong_joints else 0 for joint in ALL_JOINTS]

#프레임 라벨 리스트 → 시퀀스 라벨 벡터
def get_sequence_label(frame_labels):
    return np.max(frame_labels, axis=0)

#PyTorch Dataset 클래스
class SkeletonSequenceDataset(Dataset):
    def __init__(self, root_dir, sequence_length=30, transform=None):
        self.root_dir = root_dir
        self.sequence_length = sequence_length
        self.transform = transform if transform else transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.ToTensor()
        ])
        self.samples = []
        self._load_samples()

    def _load_samples(self):
        for folder in os.listdir(self.root_dir):
            folder_path = os.path.join(self.root_dir, folder)
            if not os.path.isdir(folder_path):
                continue
            label_path = os.path.join(folder_path, "labels.json")
            if not os.path.exists(label_path):
                continue

            with open(label_path, "r") as f:
                label_data = json.load(f)

            frame_labels = {
                item["frame_idx"]: joints_to_vector(item["wrong_joints"])
                for item in label_data
            }

            image_files = sorted([
                f for f in os.listdir(folder_path)
                if f.endswith(".png") and f.startswith("frame_")
            ])

            for i in range(0, len(image_files) - self.sequence_length + 1, self.sequence_length):
                seq_images = image_files[i:i+self.sequence_length]
                seq_labels = [
                    frame_labels.get(i + idx, [0]*12) for idx in range(self.sequence_length)
                ]
                final_label = get_sequence_label(seq_labels)
                self.samples.append((folder_path, seq_images, final_label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
      folder_path, seq_images, label = self.samples[idx]
      images = [
          self.transform(Image.open(os.path.join(folder_path, img)).convert("RGB"))
          for img in seq_images
      ]
      images = torch.stack(images)
      return images, torch.tensor(label, dtype=torch.float32)

In [6]:
class PreprocessedTensorDataset(torch.utils.data.Dataset):
    def __init__(self, tensor_dir):
        self.paths = sorted([
            os.path.join(tensor_dir, f)
            for f in os.listdir(tensor_dir)
            if f.endswith(".pt")
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        return torch.load(self.paths[idx])


In [7]:
#모델정의 시작
class CNNLSTMModel(nn.Module):
    def __init__(self, hidden_dim=128, num_classes=12, lstm_layers=1):
        super(CNNLSTMModel, self).__init__()
        #CNN: ResNet18
        base_model = models.resnet18(weights=None)
        self.cnn_backbone = nn.Sequential(*list(base_model.children())[:-1])  # (B, 512, 1, 1)
        self.cnn_output_dim = 512
        #LSTM
        self.lstm = nn.LSTM(input_size=self.cnn_output_dim,
                            hidden_size=hidden_dim,
                            num_layers=lstm_layers,
                            batch_first=True)
        #FC
        self.classifier = nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(-1, C, H, W)  # (B*T, C, H, W)
        features = self.cnn_backbone(x)  # (B*T, 512, 1, 1)
        features = features.view(B, T, -1)  # (B, T, 512)

        lstm_out, _ = self.lstm(features)
        last_hidden = lstm_out[:, -1, :]  # (B, hidden_dim)

        out = self.classifier(last_hidden)  # (B, 12)
        return out

In [8]:
#평가함수
def compute_metrics(preds, labels, threshold=0.5):
    preds = (torch.sigmoid(preds) > threshold).int().cpu().numpy()
    labels = labels.int().cpu().numpy()
    precision = precision_score(labels, preds, average=None, zero_division=0)
    recall = recall_score(labels, preds, average=None, zero_division=0)
    accuracy = accuracy_score(labels, preds)
    return accuracy, precision, recall

In [ ]:
#학습+검증 루프
import torch
from torch.utils.data import random_split, DataLoader
from tqdm import tqdm
import torch.optim as optim

#하이퍼파라미터
BATCH_SIZE = 4
#Hundred -> 20에폭
#HipCircle -> 28에폭
#Teaser -> 에폭
#Swimming -> 에폭
EPOCHS = 20
LEARNING_RATE = 1e-4
SEQUENCE_LENGTH = 30

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNLSTMModel().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 데이터셋 분할
# 자세 이름 변경
full_dataset = PreprocessedTensorDataset('/content/processed_tensors_Teaser')
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


#학습 루프
model = CNNLSTMModel().to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
device = next(model.parameters()).device

train_losses = []
val_accuracies = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    print(f"Epoch {epoch+1} - Train Loss: {avg_train_loss:.4f}")

    # 검증
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            all_preds.append(outputs)
            all_labels.append(y)

    all_preds = torch.cat(all_preds, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    acc, precision, recall = compute_metrics(all_preds, all_labels)
    val_accuracies.append(acc)
    print(f"Val Accuracy: {acc:.4f}")

    # 저장
    torch.save(model.state_dict(), f"cnn_lstm_epoch{epoch+1}.pth")


In [ ]:
#테스트
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        outputs = model(x)
        all_preds.append(outputs)
        all_labels.append(y)

all_preds = torch.cat(all_preds, dim=0)
all_labels = torch.cat(all_labels, dim=0)
acc, precision, recall = compute_metrics(all_preds, all_labels)
print("\n최종 테스트 성능")
print(f"Accuracy: {acc:.4f}")
for i, joint in enumerate(ALL_JOINTS):
    print(f"{joint:10s} | Precision: {precision[i]:.2f} | Recall: {recall[i]:.2f}")

#시각화
plt.plot(train_losses, label='Train Loss')
plt.plot(val_accuracies, label='Val Accuracy')
plt.xlabel("Epoch")
plt.legend()
plt.title("Training Loss & Validation Accuracy")
plt.show()


In [ ]:
#예측 결과 CSV로 저장
import pandas as pd

# sigmoid 후 0.5 이상이면 1
final_preds = (torch.sigmoid(all_preds) > 0.5).int().cpu().numpy()
final_labels = all_labels.int().cpu().numpy()

results = []
for i in range(len(final_preds)):
    pred = final_preds[i]
    true = final_labels[i]
    row = {
        "sample_idx": i,
        **{f"pred_{j}": pred[j] for j in range(12)},
        **{f"true_{j}": true[j] for j in range(12)}
    }
    results.append(row)

df = pd.DataFrame(results)
df.to_csv("test_predictions.csv", index=False)
print("예측 결과 test_predictions.csv 저장 완료")